In [1]:
import pandas as pd
import altair as alt

df = pd.read_csv("student_data_cleaned.csv")

In [2]:
# Altair 1: Boxplot of Study Hours Per Day by Major

boxplot = alt.Chart(df).mark_boxplot().encode(
    y = alt.Y("Major:N", title="Major"),
    x = alt.X("Study_Hours_Per_Day:Q", title="Study Hours Per Day"),
    color = alt.Color("Major:N", legend=None),
    tooltip = alt.Tooltip(["Major", "Study_Hours_Per_Day:Q"])
).properties(
    title = "Student Study Hours Per Day by Major",
    width=500,
    height=300
)

boxplot.save("boxplot_study_hours_per_day_by_major.html")

This chart displays the distribution of daily study hours across six majors to explore whether a student's field of study influences how much time they dedicate to studying. The interquartile ranges are very similar across all majors, with the middle 50% of students in every major studying between around 2.5 and 6 hours per day. The median study hours are nearly identical as well, suggesting that no single major stands out as significantly more or less demanding in terms of time spent studying. However, all majors show a notable number of outliers on the higher end, with some students studying up to 14 hours per day regardless of their major. This pattern suggests that extreme study habits are not unique to any one major. Overall, the visualization indicates that daily study hours are mainly consistent across majors, and that individual student habits may play a larger role in determining study time than the major itself.

In [ ]:
# Altair 2: Scatter plot of Sleep Hours vs GPA Change with linked bar chart

# Radio button for major type selection
input_radio = alt.binding_radio(
    options=[None, "Technical", "Non-technical"],
    labels=["Both", "Technical", "Non-technical"],
    name="Major Type: "
)
type_select = alt.selection_point(fields=["Major_Type"], bind=input_radio)

# Age slider for filtering by age
age_slider = alt.binding_range(min=18, max=24, step=1, name="Max Age: ")
age_param = alt.param(value=24, bind=age_slider)

# Brush for linked bar
brush = alt.selection_interval()

scatter = alt.Chart(df).mark_circle(opacity=0.5, size=40).encode(
    x=alt.X("Sleep_Hours_Per_Day:Q", title="Sleep Hours"),
    y=alt.Y("GPA_Change:Q", title="GPA Change"),
    color=alt.condition(
        brush,
        alt.Color("Major_Type:N", scale=alt.Scale(
            domain=["Technical", "Non-technical"],
            range=["#0D55F0", "#D7274A"]
        )),
        alt.value("lightgrey")
    ),
    tooltip=[
        alt.Tooltip("Major:N"),
        alt.Tooltip("Previous_GPA:Q", title="Previous GPA", format=".2f"),
        alt.Tooltip("Final_CGPA:Q", title="Final CGPA", format=".2f"),
        alt.Tooltip("GPA_Change:Q", title="GPA Change", format=".2f"),
        alt.Tooltip("Age:Q")
    ]
).add_params(
    brush, age_param, type_select
).transform_filter(
    alt.datum.Age <= age_param
).transform_filter(
    type_select
).properties(
    title="Sleep Hours vs GPA Change",
    width=550,
    height=350
)

bars = alt.Chart(df).mark_bar().encode(
    x=alt.X("Major:N", title="Major"),
    y=alt.Y("count():Q", title="Count of Students"),
    color=alt.Color("Major_Type:N", scale=alt.Scale(
        domain=["Technical", "Non-technical"],
        range=["#0D55F0", "#D7274A"]), 
        legend=alt.Legend(title="Major Type"))
).transform_filter(
    brush
).transform_filter(
    alt.datum.Age <= age_param
).transform_filter(
    type_select
).properties(
    title="Students in Selected Region",
    width=550,
    height=150
)

(scatter & bars).save("scatter_bar_linked_sleep_gpa.html")

This interactive chart explores the relationship between sleep hours and GPA change from a student's previous to final Cumulative GPA, with points colored by whether the student belongs to a technical or non-technical major. The slider allows filtering by age and the radio button allows toggling between major types, while the brush selection links to a bar chart below showing the count of students in the selected region. Overall, the scatter plot shows no strong directional trend between sleep hours and GPA change. Students across all sleep ranges show both positive and negative GPA changes, suggesting that sleep alone is not a reliable predictor of academic improvement. Both technical and non-technical majors are distributed across the plot with significant overlap, indicating that major type does not create a clear divide in how sleep relates to performance. The linked bar chart shows that when a region of the scatter plot is selected, the distribution of students across majors is relatively even, reinforcing that no single major dominates any particular combination of sleep hours and GPA change.

In [4]:
# Altair 3: Bar chart of Average Attendance Rate by Major Type

bar = alt.Chart(df).mark_bar().encode(
    x = alt.X("Major:N", title="Major"),
    y = alt.Y("mean(Attendance_Pct):Q", title="Average Attendance (%)"),
    color = alt.Color("Major_Type:N", 
                      scale=alt.Scale(domain=["Technical", "Non-technical"], range=["#0D55F0", "#D7274A"]),
                      legend=alt.Legend(title="Major Type")
    ),
    tooltip=[
        alt.Tooltip("Major:N", title="Major"),
        alt.Tooltip("Major_Type:N", title="Major Type"),
        alt.Tooltip("mean(Attendance_Pct):Q", title="Avg Attendance (%)", format=".1f")
    ]
).properties(
    title = "Average Attendance Rate by Major Type",
    width=250,
    height=450
)

bar.save("bar_avg_attendance_by_major_type.html")

This chart compares the average attendance rate across all six majors, grouped and colored by major type, to investigate whether technical or non-technical students show different attendance behaviors. The most striking finding is how uniform attendance rates are across all majors, with every major averaging between 83% and 85%. Engineering students have the highest average attendance while Computer Science students have the lowest among technical majors, although the differences are minimal. Among non-technical majors, Economics and Psychology shows slightly higher attendance than Business. The consistency across both technical and non-technical majors suggests that attendance habits are not strongly influenced by a student's field of study. This uniformity may reflect a shared academic culture where attendance expectations are similar regardless of major, or it may indicate that the factors prompting attendance, such as personal motivation or course structure, are mainly independent of the subject being studied.